# Cálculo de ELO por equipo

Procesa los partidos de 2025 y 2026 del archivo `odds.csv` y calcula el ELO actual de cada equipo.

**Parámetros:**
- ELO inicial: 1500
- Ventaja de localía: +100 puntos
- K-factor: 30
- Resultado: Victoria=1, Empate=0.5, Derrota=0

In [ ]:
import pandas as pd
import numpy as np

## 1. Carga y filtrado de datos

In [ ]:
df = pd.read_csv('../public/data/odds.csv', encoding='utf-8-sig')

# Filtrar solo 2025 y 2026
df = df[df['Season'].astype(str).isin(['2025', '2026'])].copy()

print(f'Partidos 2025-2026: {len(df)}')
print(df['Season'].value_counts())

In [ ]:
# Lista de los 30 equipos (nombres tal como aparecen en el CSV)
equipos_csv = [
    'Ind. Rivadavia', 'Estudiantes L.P.', 'Boca Juniors', 'River Plate',
    'Argentinos Jrs', 'Velez Sarsfield', 'Rosario Central', 'Talleres Cordoba',
    'Belgrano', 'Gimnasia L.P.', 'Independiente', 'Lanus', 'Huracan',
    'San Lorenzo', 'Union de Santa Fe', 'Racing Club', 'Instituto',
    'Barracas Central', 'Tigre', 'Defensa y Justicia', 'Sarmiento Junin',
    'Gimnasia Mendoza', 'Banfield', 'Platense', 'Central Cordoba',
    'Newells Old Boys', 'Atl. Tucuman', 'Dep. Riestra', 'Aldosivi',
    'Estudiantes Rio Cuarto'
]

# Filtrar solo partidos donde ambos equipos están en la lista
df = df[df['Home'].isin(equipos_csv) & df['Away'].isin(equipos_csv)].copy()

print(f'Partidos entre los 30 equipos: {len(df)}')

In [ ]:
# Ordenar cronológicamente
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df = df.sort_values('Date').reset_index(drop=True)

print(f'Primer partido: {df["Date"].min().date()}')
print(f'Último partido: {df["Date"].max().date()}')
df.head()

## 2. Cálculo de ELO

In [ ]:
K = 30
HOME_ADVANTAGE = 100
ELO_INICIAL = 1500

elo = {equipo: ELO_INICIAL for equipo in equipos_csv}

for _, partido in df.iterrows():
    home = partido['Home']
    away = partido['Away']
    resultado = partido['Res']  # H, D, A

    # Expectativa con ventaja de localía
    diff = (elo[home] + HOME_ADVANTAGE) - elo[away]
    E_home = 1 / (1 + 10 ** (-diff / 400))
    E_away = 1 - E_home

    # Resultado real
    if resultado == 'H':
        S_home, S_away = 1.0, 0.0
    elif resultado == 'D':
        S_home, S_away = 0.5, 0.5
    else:  # A
        S_home, S_away = 0.0, 1.0

    # Actualizar ELO
    elo[home] += K * (S_home - E_home)
    elo[away] += K * (S_away - E_away)

print('ELO calculado para todos los equipos.')

## 3. Resultado

In [ ]:
# Mapeo nombre CSV → id y nombre display
equipos_map = {
    'Ind. Rivadavia':        {'id': 'independiente-rivadavia', 'nombre': 'Ind. Rivadavia'},
    'Estudiantes L.P.':      {'id': 'estudiantes-lp',          'nombre': 'Estudiantes'},
    'Boca Juniors':          {'id': 'boca',                    'nombre': 'Boca Jrs.'},
    'River Plate':           {'id': 'river',                   'nombre': 'River'},
    'Argentinos Jrs':        {'id': 'argentinos',              'nombre': 'Argentinos'},
    'Velez Sarsfield':       {'id': 'velez',                   'nombre': 'Vélez'},
    'Rosario Central':       {'id': 'central',                 'nombre': 'Central'},
    'Talleres Cordoba':      {'id': 'talleres',                'nombre': 'Talleres'},
    'Belgrano':              {'id': 'belgrano',                'nombre': 'Belgrano'},
    'Gimnasia L.P.':         {'id': 'gimnasia',                'nombre': 'Gimnasia'},
    'Independiente':         {'id': 'independiente',           'nombre': 'Independiente'},
    'Lanus':                 {'id': 'lanus',                   'nombre': 'Lanús'},
    'Huracan':               {'id': 'huracan',                 'nombre': 'Huracán'},
    'San Lorenzo':           {'id': 'san-lorenzo',             'nombre': 'San Lorenzo'},
    'Union de Santa Fe':     {'id': 'union',                   'nombre': 'Unión'},
    'Racing Club':           {'id': 'racing',                  'nombre': 'Racing'},
    'Instituto':             {'id': 'instituto',               'nombre': 'Instituto'},
    'Barracas Central':      {'id': 'barracas',                'nombre': 'Barracas'},
    'Tigre':                 {'id': 'tigre',                   'nombre': 'Tigre'},
    'Defensa y Justicia':    {'id': 'defensa',                 'nombre': 'Defensa'},
    'Sarmiento Junin':       {'id': 'sarmiento',               'nombre': 'Sarmiento'},
    'Gimnasia Mendoza':      {'id': 'gimnasia-m',              'nombre': 'Gimnasia (M)'},
    'Banfield':              {'id': 'banfield',                'nombre': 'Banfield'},
    'Platense':              {'id': 'platense',                'nombre': 'Platense'},
    'Central Cordoba':       {'id': 'central-cordoba',         'nombre': 'Central Córdoba'},
    'Newells Old Boys':      {'id': 'newells',                 'nombre': "Newell's"},
    'Atl. Tucuman':          {'id': 'atl-tucuman',             'nombre': 'Atl. Tucumán'},
    'Dep. Riestra':          {'id': 'riestra',                 'nombre': 'Riestra'},
    'Aldosivi':              {'id': 'aldosivi',                'nombre': 'Aldosivi'},
    'Estudiantes Rio Cuarto':{'id': 'estudiantes-rc',          'nombre': 'Estudiantes RC'},
}

# Armar DataFrame ordenado por ELO descendente
filas = []
for nombre_csv, elo_val in sorted(elo.items(), key=lambda x: -x[1]):
    info = equipos_map[nombre_csv]
    filas.append({
        'id':        info['id'],
        'nombre':    info['nombre'],
        'nombreCSV': nombre_csv,
        'elo':       round(elo_val, 1)
    })

df_elo = pd.DataFrame(filas)
df_elo

In [ ]:
df_elo.to_csv('../public/data/elo-equipos.csv', index=False, encoding='utf-8')
print('Archivo elo-equipos.csv generado en public/data/')